In [1]:
import os
import numpy as np
import pandas as pd
import pydicom
%matplotlib inline
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model
from PIL import Image

In [4]:
# This function takes the numpy array output by check_dicom and
# runs the appropriate pre-processing needed for our model input
def preprocess_image(img, img_mean, img_std, img_size):
    # Normalize using the same preprocessing as training (rescale by 1/255)
    proc_img = img / 255.0
    
    # Convert grayscale to 3 channels if needed
    if proc_img.ndim == 2:
        proc_img = np.repeat(proc_img[..., np.newaxis], 3, axis=-1)
    
    # Properly resize the image using PIL instead of np.resize
    if proc_img.shape[:2] != img_size[1:3]:
        # Convert to PIL Image, resize, and convert back
        pil_img = Image.fromarray((proc_img[:, :, 0] * 255).astype(np.uint8))
        # Use Image.LANCZOS for compatibility with older Pillow versions
        pil_img = pil_img.resize((img_size[2], img_size[1]), Image.LANCZOS)
        proc_img_resized = np.array(pil_img) / 255.0
        proc_img = np.repeat(proc_img_resized[..., np.newaxis], 3, axis=-1)
    
    return np.expand_dims(proc_img, axis=0)

In [5]:
test_dicoms = ['test1.dcm', 'test2.dcm', 'test3.dcm', 'test4.dcm', 'test5.dcm', 'test6.dcm']

# Point to the actual trained model artifacts from Build and train model.ipynb
model_json_path = './my_model.json'  # Model architecture
weight_path = './xray_class_my_model.best.hdf5'  # Best weights from training

IMG_SIZE = (1, 224, 224, 3)  # VGG16 input size

# Match the training preprocessing: rescale by 1/255
# (mean and std are not actually used with our preprocessing, but kept for compatibility)
img_mean = 0.0
img_std = 1.0

# Use the F1-optimized threshold from training evaluation
# Update this with your actual best_threshold value from the training notebook
thresh = 0.5  # Replace with best_threshold from F1 optimization

try:
    my_model = load_trained_model(model_json_path, weight_path)
except FileNotFoundError as e:
    print(e)
    my_model = None

# Use the .dcm files to test your prediction
print('Running inference on test DICOM files...\n')
for i in test_dicoms:
    img = check_dicom(i)

    if img is None:
        continue

    img_proc = preprocess_image(img, img_mean, img_std, IMG_SIZE)
    pred = predict_image(my_model, img_proc, thresh)
    
    result = 'PNEUMONIA DETECTED' if pred == 1 else 'No pneumonia'
    print('{}: {} (prediction score: {:.3f})\n'.format(i, result, pred))

Loading weights from: ./xray_class_my_model.best.hdf5
Running inference on test DICOM files...

Load file test1.dcm ...
test1.dcm: No pneumonia (prediction score: 0.000)

Load file test2.dcm ...
test2.dcm: No pneumonia (prediction score: 0.000)

Load file test3.dcm ...
test3.dcm: No pneumonia (prediction score: 0.000)

Load file test4.dcm ...
test4.dcm: No pneumonia (prediction score: 0.000)

Load file test5.dcm ...
test5.dcm: No pneumonia (prediction score: 0.000)

Load file test6.dcm ...
test6.dcm: No pneumonia (prediction score: 0.000)

